In [ ]:
# =====================================================================
# MODELS SETUP
# =====================================================================

import torch
import cv2
import clip
import numpy as np
import pandas as pd
from PIL import Image
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator


# Load CLIP model for semantic matching
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

# Load SAM model for image segmentation
sam_checkpoint = "sam_vit_h_4b8939.pth"
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
sam.to(device)

# Configure the automatic mask generator
mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.95,
    min_mask_region_area=500
)

In [ ]:
# =====================================================================
# COMPUTER VISION FUNCTIONS (SAM + CLIP + Statistics)
# =====================================================================

def load_image(path):
    """
    Loads an image using OpenCV and converts color space from BGR to RGB.
    """
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def sam_segment(image):
    """
    Generates segmentation masks for the given image using SAM.
    """
    return mask_generator.generate(image)


def clip_score(mask, image, labels):
    """
    Calculates CLIP similarity scores between an isolated mask area and target text labels.
    """
    masked = image.copy()
    masked[~mask] = 0  # Black out everything outside the mask to isolate the object

    pil_img = Image.fromarray(masked)
    img_input = clip_preprocess(pil_img).unsqueeze(0).to(device)
    text_input = clip.tokenize(labels).to(device)

    with torch.no_grad():
        img_feat = clip_model.encode_image(img_input)
        txt_feat = clip_model.encode_text(text_input)
        scores = (img_feat @ txt_feat.T).softmax(dim=-1)

    return scores.cpu().numpy()[0]


def select_object_masks(image, masks, labels):
    """
    Assigns the most compatible SAM mask to each text label using CLIP scores.
    """
    best = {label: (None, -1) for label in labels}

    for m in masks:
        seg = m["segmentation"]
        scores = clip_score(seg, image, labels)

        for i, label in enumerate(labels):
            if scores[i] > best[label][1]:
                best[label] = (m, scores[i])

    return {k: v[0] for k, v in best.items()}


def mask_stats(mask, img_shape):
    """
    Extracts spatial statistics (area, centroid, bounding box) from a mask.
    """
    seg = mask["segmentation"].astype(np.uint8)
    area = np.sum(seg)

    # Calculate spatial center mass (centroid)
    ys, xs = np.where(seg)
    cx, cy = xs.mean(), ys.mean()

    h, w, _ = img_shape
    dist_center = np.sqrt((cx - w/2)**2 + (cy - h/2)**2)

    return {
        "area": area,
        "center_dist": dist_center,
        "bbox": mask["bbox"],
        "centroid": (cx, cy)
    }


def object_dominance(stats_A, stats_B, alpha=0.5):
    """
    Computes the Object Dominance Score (ODS) based on area and center proximity ratios.
    """
    area_ratio = stats_A["area"] / (stats_B["area"] + 1e-6)
    dist_ratio = stats_B["center_dist"] / (stats_A["center_dist"] + 1e-6)
    return alpha * area_ratio + (1 - alpha) * dist_ratio


def analyze_image(image_path, obj_A, obj_B):
    """
    Runs the full CV pipeline to isolate objects and compute visual properties.
    """
    image = load_image(image_path)
    masks = sam_segment(image)

    selected = select_object_masks(image, masks, [obj_A, obj_B])
    if selected[obj_A] is None or selected[obj_B] is None:
        return None

    sA = mask_stats(selected[obj_A], image.shape)
    sB = mask_stats(selected[obj_B], image.shape)
    ods = object_dominance(sA, sB)

    return ods, sA, sB

In [ ]:
# =====================================================================
# SPATIAL GEOMETRY VERIFICATION
# =====================================================================

def check_relation(relation, A, B):
    """
    Verifies if the spatial relation between Object A (First Token)
    and Object B (Second Token) holds geometrically using centroids.
    """
    cxA, cyA = A["centroid"]
    cxB, cyB = B["centroid"]

    relation = relation.strip().lower()

    # --- Vertical Axis ---
    if relation == "on top of":
        return cyA < cyB  # In pixels, lower Y values mean higher position

    if relation == "under":
        return cyA > cyB

    # --- Horizontal Axis ---
    if "left" in relation:
        return cxA < cxB

    if "right" in relation:
        return cxB < cxA

    # --- Proximity ---
    if "next to" in relation:
        # Objects must be aligned predominantly on the horizontal plane
        return abs(cxA - cxB) > abs(cyA - cyB)

    return False

In [ ]:
# =====================================================================
# PIPELINE EXECUTION
# =====================================================================
import os
import pandas as pd

IMAGES_DIR      = "./images"                            # path to the folder with the images
CSV_INPUT_PATH  = "./results/res_wo.csv"                # path to the word order vlm question answering results
CSV_OUTPUT_PATH = "./results/cv_word_order_final.csv"   # where this analysis results will be saved

def process_word_order_results(csv_path, IMAGES_DIR):
    # Read the CSV
    df = pd.read_csv(csv_path)
    results = []

    for i, row in df.iterrows():
        # Build the exact path to the image
        image_path = os.path.join(IMAGES_DIR, row["img_name"])

        if not os.path.exists(image_path):
            print(f"Skipping row {i}: Image file not found at {image_path}")
            continue

        # Run core CV extraction (SAM + CLIP matching)
        out = analyze_image(image_path, row["object_1"], row["object_2"])
        if out is None:
            print(f"Skipping row {i} ({row['img_name']}): SAM/CLIP failed to isolate targets.")
            continue

        ods, sA, sB = out

        # Verify if the real image geometry matches the prompt intent
        is_correct_geometry = check_relation(row["relation"], sA, sB)

        # Handle formatting inconsistencies in VLM answers (e.g., 'Yes', 'yes', 'no.')
        vlm_ans = str(row["a3"]).strip().lower().replace('.', '')

        # Merge original data with computed CV features
        results.append({
            "prompt_index": row["prompt_index"],
            "img_name": row["img_name"],
            "prompt": row["prompt"],
            "object_1": row["object_1"],
            "object_2": row["object_2"],
            "relation": row["relation"],
            "ODS": ods,
            "spatial_error": int(not is_correct_geometry),
            "vlm_sees_obj1": row["a1"],
            "vlm_sees_obj2": row["a2"],
            "vlm_spatial_prediction": vlm_ans
        })

        if i % 10 == 0:
            print(f"Successfully evaluated {i} images...")

    return pd.DataFrame(results)

# Execute the process starting from row 0
cv_word_order = process_word_order_results(CSV_INPUT_PATH, IMAGES_DIR)

# Save results directly into the shared workspace 'results' directory
cv_word_order.to_csv(CSV_OUTPUT_PATH, index=False)

In [ ]:
#ANALYSIS OF THE RESULTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load the generated dataset
df = pd.read_csv(CSV_OUTPUT_PATH)

# Clean VLM text strings for evaluation (lowercase and strip text)
df["vlm_sees_obj1"] = df["vlm_sees_obj1"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["vlm_sees_obj2"] = df["vlm_sees_obj2"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["vlm_spatial_prediction"] = df["vlm_spatial_prediction"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)

# =====================================================================
# 1. VISUAL BIAS ANALYSIS
# =====================================================================

# Outlier filtering based on IQR
Q1 = df["ODS"].quantile(0.25)
Q3 = df["ODS"].quantile(0.75)
IQR = Q3 - Q1

df_filt = df[
    (df["ODS"] >= Q1 - 1.5 * IQR) &
    (df["ODS"] <= Q3 + 1.5 * IQR)
].copy()

# Apply log transform to balance the ratio
df_filt["log_ODS"] = np.log(df_filt["ODS"])

# Group by the native prompt_index to verify symmetric image pairs
pairs = (
    df_filt
    .groupby("prompt_index")["log_ODS"]
    .apply(list)
    .reset_index()
)
# Ensure we only keep fully completed pairs (both combinations generated)
pairs = pairs[pairs["log_ODS"].apply(len) == 2]

def classify_first_noun_bias(log_ods_pair):
    o1, o2 = log_ods_pair
    # First-noun bias occurs if the model draws the first named object
    # larger/closer to the center in BOTH symmetrical prompt variants
    return int(o1 > 0 and o2 > 0)

pairs["first_noun_bias"] = pairs["log_ODS"].apply(classify_first_noun_bias)
bias_rate = pairs["first_noun_bias"].mean()

print("="*50)
print("             VISUAL BIAS METRICS (ODS)            ")
print("="*50)
print(f"First-Noun Bias Rate (Visual Dominance): {bias_rate:.2%}")

# =====================================================================
# 2. LINGUISTIC VS GEOMETRIC ALIGNMENT (New VLM Variables Analysis)
# =====================================================================

# A VLM response is strictly correct only if it answers 'yes' to the prompt configuration
df["vlm_is_correct"] = df["vlm_spatial_prediction"] == "yes"

# Compute the correlation between real spatial errors (from SAM) and VLM predictions
vlm_accuracy_total = df["vlm_is_correct"].mean()

# How often does the VLM fail because the generator actually botched the image geometry?
mismatch_mask = (df["spatial_error"] == 1) & (df["vlm_is_correct"] == True)
vlm_fooled_by_geometry_rate = mismatch_mask.mean()

print("\n" + "="*50)
print("         VLM TEXTUAL VS GEOMETRIC ALIGNMENT       ")
print("="*50)
print(f"Overall VLM Prompt Accuracy: {vlm_accuracy_total:.2%}")
print(f"VLM Fooled by Generation Errors: {vlm_fooled_by_geometry_rate:.2%}")
print(f"  -> (Instances where VLM said 'yes' but SAM proved the geometry was wrong)")

# Let's see Object Hallucination rates
obj1_missing = (df["vlm_sees_obj1"] == "no").mean()
obj2_missing = (df["vlm_sees_obj2"] == "no").mean()
print(f"Object 1 Hallucination Rate (VLM claimed missing): {obj1_missing:.2%}")
print(f"Object 2 Hallucination Rate (VLM claimed missing): {obj2_missing:.2%}")

In [ ]:
#STATISTICAL ANALYSIS

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, chi2_contingency
from statsmodels.stats.contingency_tables import mcnemar

# Load the dataset
df = pd.read_csv("results/cv_word_order_final.csv")

# Clean strings for exact categorical matching
df["vlm_sees_obj1"] = df["vlm_sees_obj1"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["vlm_sees_obj2"] = df["vlm_sees_obj2"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["vlm_spatial_prediction"] = df["vlm_spatial_prediction"].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)
df["vlm_is_correct"] = df["vlm_spatial_prediction"] == "yes"

print("="*60)
print("             ADVANCED STATISTICAL INFERENCE             ")
print("="*60)

# =====================================================================
# TEST 1: WILCOXON SIGNED-RANK TEST (On Paired log_ODS)
# =====================================================================
print("\n[1] TEST DI WILCOXON SUL BIAS GEOMETRICO (log_ODS)")

# Filter outliers and apply log transform exactly like your script
Q1 = df["ODS"].quantile(0.25)
Q3 = df["ODS"].quantile(0.75)
IQR = Q3 - Q1
df_filt = df[(df["ODS"] >= Q1 - 1.5 * IQR) & (df["ODS"] <= Q3 + 1.5 * IQR)].copy()
df_filt["log_ODS"] = np.log(df_filt["ODS"])

# Sort to guarantee that row 0 (Prompt A) and row 1 (Prompt B) match perfectly inside the group
df_filt = df_filt.sort_values(by=["prompt_index", "img_name"])

# Extract pairs
paired_data = df_filt.groupby("prompt_index")["log_ODS"].apply(list).reset_index()
paired_data = paired_data[paired_data["log_ODS"].apply(len) == 2]

# Split into two vectors: Prompt A (original) and Prompt B (symmetrical inversion)
vec_A = np.array([x[0] for x in paired_data["log_ODS"]])
vec_B = np.array([x[1] for x in paired_data["log_ODS"]])

# Run Wilcoxon Signed-Rank Test
stat_wilc, p_val_wilc = wilcoxon(vec_A, vec_B)
print(f"  -> Wilcoxon Statistic: {stat_wilc:.4f}")
print(f"  -> p-value: {p_val_wilc:.5e}")
if p_val_wilc < 0.05:
    print("  ->SIGNIFICANT RESULT: There is a systematic geometric difference when we reverse the order of the words.")
else:
    print("  -> NOT SIGNIFICANT: The geometric difference between the two variants could be random.")


# =====================================================================
# TEST 2: CHI-SQUARE TEST OF INDEPENDENCE (VLM Fooled by Geometry)
# =====================================================================
print("\CHI-SQUARE TEST: Is the VLM influenced by FLUX errors?")

# Create a 2x2 contingency table: VLM Accuracy vs Real Spatial Error
# Columns: Real Spatial Error (0 = Correct Geometry, 1 = Error Geometry)
# Rows: VLM Prediction (True = Said 'Yes', False = Said 'No')
contingency_matrix = pd.crosstab(df["vlm_is_correct"], df["spatial_error"])

print("  -> Contingency Table (VLM vs SAM Geometry):")
print(contingency_matrix)

# Run Chi-Square
chi2, p_val_chi2, dof, expected = chi2_contingency(contingency_matrix)
print(f"  -> Chi-Square Statistic: {chi2:.4f}")
print(f"  -> p-value: {p_val_chi2:.5e}")
if p_val_chi2 < 0.05:
    print("  -> SIGNIFICANT RESULT: The VLM responses are significantly dependent on / influenced by the geometric errors in the image.")
else:
    print("  -> NOT SIGNIFICANT: The VLM responses do not show a robust statistical dependence on the actual geometry.")


# =====================================================================
# TEST 3: MCNEMAR'S TEST (Object 1 vs Object 2 Hallucinations)
# =====================================================================
print("\n[3] MCNEMAR TEST: Is the hallucination rate different between Object 1 and Object 2?")

# Convert 'yes'/'no' detections into boolean missing flags (True if missing/hallucinated)
obj1_missing = df["vlm_sees_obj1"] == "no"
obj2_missing = df["vlm_sees_obj2"] == "no"

# Create a 2x2 matrix for paired proportions
# Rows: Obj1 Missing (True/False), Columns: Obj2 Missing (True/False)
mcnemar_table = pd.crosstab(obj1_missing, obj2_missing)

print("  -> Contingency Table (Missing Obj1 vs Missing Obj2):")
print(mcnemar_table)

# Run McNemar's Test (using exact binomial distribution because of small cell sizes)
res_mcnemar = mcnemar(mcnemar_table, exact=True)
print(f"  -> p-value McNemar: {res_mcnemar.pvalue:.5e}")
if res_mcnemar.pvalue < 0.05:
    print("  -> SIGNIFICANT RESULT: The model suffers significantly more from hallucinations of the second object than of the first.")
else:
    print("  -> NOT SIGNIFICANT: The difference in hallucination rate between the two objects is not statistically significant")
print("="*60)

In [ ]:
# ============================================================
# CONTINUOUS EFFECT ANALYSIS (PAIRWISE DELTA + CI + EFFECT SIZE)
# ============================================================

import numpy as np
from scipy.stats import bootstrap, wilcoxon

print("=" * 70)
print("        CONTINUOUS FIRST-NOUN EFFECT ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Compute pairwise continuous difference
# ------------------------------------------------------------
# vec_A = log_ODS of original prompt order
# vec_B = log_ODS of inverted prompt order
#
# Positive delta:
# -> original ordering produces stronger dominance
# Negative delta:
# -> inverted ordering produces stronger dominance

deltas = vec_A - vec_B

mean_delta = np.mean(deltas)

# ------------------------------------------------------------
# 2. Bootstrap 95% Confidence Interval (BCa)
# ------------------------------------------------------------
# BCa bootstrap is robust for non-parametric paired data

bootstrap_result = bootstrap(
    (deltas,),
    np.mean,
    confidence_level=0.95,
    method="BCa",
    random_state=42
)

ci_lower = bootstrap_result.confidence_interval.low
ci_upper = bootstrap_result.confidence_interval.high

# ------------------------------------------------------------
# 3. Wilcoxon Signed-Rank Test
# ------------------------------------------------------------
wilc_result = wilcoxon(vec_A, vec_B)

W = wilc_result.statistic
p_value = wilc_result.pvalue

# ------------------------------------------------------------
# 4. Rank-Biserial Correlation (Effect Size)
# ------------------------------------------------------------
# Appropriate effect size for Wilcoxon paired comparisons

N = len(vec_A)

rank_biserial = 1 - (
    (2 * W) / (N * (N + 1) / 2)
)

# ------------------------------------------------------------
# 5. Print Results
# ------------------------------------------------------------
print(f"Number of paired prompts analysed: {N}")
print(f"Mean Δ(log_ODS_A - log_ODS_B): {mean_delta:.4f}")
print(f"95% BCa Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Wilcoxon Signed-Rank p-value: {p_value:.5e}")
print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

# ------------------------------------------------------------
# 6. Interpretation
# ------------------------------------------------------------

print("\nINTERPRETATION")

if p_value < 0.05:
    print("-> Prompt inversion produces a statistically significant change in object dominance.")
else:
    print("-> No statistically significant dominance shift was detected.")

if ci_lower > 0:
    print("-> The confidence interval excludes zero and is entirely positive.")
    print("   This suggests a stable directional shift favouring the first-mentioned noun.")
elif ci_upper < 0:
    print("-> The confidence interval excludes zero and is entirely negative.")
    print("   This suggests a stable directional shift favouring the second-mentioned noun.")
else:
    print("-> The confidence interval overlaps zero.")
    print("   The directional effect is not fully stable across prompt pairs.")

# Effect size interpretation
abs_r = abs(rank_biserial)

if abs_r < 0.10:
    magnitude = "negligible"
elif abs_r < 0.30:
    magnitude = "small"
elif abs_r < 0.50:
    magnitude = "moderate"
else:
    magnitude = "large"

print(f"-> Effect size magnitude: {magnitude} (rank-biserial r = {rank_biserial:.3f})")

print("=" * 70)

In [ ]:
# ==============================================================
# SECOND ANALYSIS >  FIRST-NOUN BIAS ANALYSIS (PAIRWISE, CONTINUOUS)
# ==============================================================

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, bootstrap


# --------------------------------------------------------------
# Load dataset
# --------------------------------------------------------------
df = pd.read_csv("results/cv_word_order_final.csv")

# --------------------------------------------------------------
# Reproduce preprocessing from previous analyses
# --------------------------------------------------------------
Q1 = df["ODS"].quantile(0.25)
Q3 = df["ODS"].quantile(0.75)
IQR = Q3 - Q1

df_filt = df[
    (df["ODS"] >= Q1 - 1.5 * IQR) &
    (df["ODS"] <= Q3 + 1.5 * IQR)
].copy()

# Log transform
df_filt["log_ODS"] = np.log(df_filt["ODS"])

# Keep deterministic pair ordering
df_filt = df_filt.sort_values(
    by=["prompt_index", "img_name"]
)

# --------------------------------------------------------------
# Build prompt pairs
# --------------------------------------------------------------
paired = (
    df_filt
    .groupby("prompt_index")
    .agg({
        "log_ODS": list,
        "ODS": list,
        "prompt": list
    })
    .reset_index()
)

# Keep only complete pairs
paired = paired[
    paired["log_ODS"].apply(len) == 2
].copy()

# --------------------------------------------------------------
# PART 1 — Continuous First-Noun Advantage (FNA)
# FNA = log(ODS_A) + log(ODS_B)
# --------------------------------------------------------------

paired["FNA"] = paired["log_ODS"].apply(
    lambda x: x[0] + x[1]
)

fna_values = paired["FNA"].values

# --------------------------------------------------------------
# Wilcoxon signed-rank test against 0
# H0: median(FNA) = 0
# --------------------------------------------------------------

stat_w, p_w = wilcoxon(fna_values)

# --------------------------------------------------------------
# 95% Bootstrap Confidence Interval (BCa)
# --------------------------------------------------------------

boot = bootstrap(
    (fna_values,),
    np.mean,
    confidence_level=0.95,
    method="BCa",
    random_state=42
)

ci_low = boot.confidence_interval.low
ci_high = boot.confidence_interval.high

mean_fna = np.mean(fna_values)

# --------------------------------------------------------------
# Effect size for one-sample Wilcoxon
# r = |Z| / sqrt(N)
# --------------------------------------------------------------

# Effective N used by Wilcoxon
n_effective = np.sum(fna_values != 0)

# Mean and SD of W under H0
mu_w = n_effective * (n_effective + 1) / 4

sigma_w = np.sqrt(
    n_effective *
    (n_effective + 1) *
    (2 * n_effective + 1) / 24
)

# Standardized Z
z = (stat_w - mu_w) / sigma_w

# Effect size
effect_r = abs(z) / np.sqrt(n_effective)

# --------------------------------------------------------------
# PART 2 — Strong first-noun bias pairs
# Definition:
# noun1 dominates in BOTH prompt variants
# --------------------------------------------------------------

paired["strong_first_noun_bias"] = paired["ODS"].apply(
    lambda x: (x[0] > 1) and (x[1] > 1)
)

bias_rate = paired["strong_first_noun_bias"].mean()

# --------------------------------------------------------------
# Reporting
# --------------------------------------------------------------

print("=" * 70)
print("              FIRST-NOUN BIAS ANALYSIS")
print("=" * 70)

print(f"Number of paired prompts analysed: {len(paired)}")

print("\n[1] CONTINUOUS FIRST-NOUN ADVANTAGE (FNA)")
print("-" * 70)

print(f"Mean FNA: {mean_fna:.4f}")
print(f"95% BCa Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

print(f"Wilcoxon Signed-Rank Statistic: {stat_w:.4f}")
print(f"Wilcoxon p-value: {p_w:.5e}")

print(f"Effect size (r): {effect_r:.4f}")

print("\n[2] STRONG FIRST-NOUN BIAS RATE")
print("-" * 70)

print(f"Pairs where noun1 dominates in BOTH variants: {bias_rate:.2%}")

# --------------------------------------------------------------
# Interpretation
# --------------------------------------------------------------

print("\nINTERPRETATION")
print("-" * 70)

if p_w < 0.05:
    print("-> Prompt pairs show a statistically reliable directional shift.")
else:
    print("-> No statistically reliable directional shift detected.")

if ci_low > 0:
    print("-> Confidence interval is entirely positive.")
    print("   Evidence is consistent with a systematic first-noun privileging effect.")

elif ci_high < 0:
    print("-> Confidence interval is entirely negative.")
    print("   Evidence suggests a systematic anti-first-noun tendency.")

else:
    print("-> Confidence interval overlaps zero.")
    print("   No stable directional first-noun effect detected.")

# Effect size interpretation
abs_r = abs(effect_r)

if abs_r < 0.10:
    magnitude = "negligible"
elif abs_r < 0.30:
    magnitude = "small"
elif abs_r < 0.50:
    magnitude = "moderate"
else:
    magnitude = "large"

print(
    f"-> Effect size magnitude: {magnitude} "
    f"(r = {effect_r:.3f})"
)

In [ ]:
# ==============================================================
# SECOND-NOUN BIAS ANALYSIS (PAIRWISE, CONTINUOUS)
# ==============================================================

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, bootstrap

# --------------------------------------------------------------
# Load dataset
# --------------------------------------------------------------
df = pd.read_csv("results/cv_word_order_final.csv")

# --------------------------------------------------------------
# Reproduce preprocessing from previous analyses
# --------------------------------------------------------------
Q1 = df["ODS"].quantile(0.25)
Q3 = df["ODS"].quantile(0.75)
IQR = Q3 - Q1

df_filt = df[
    (df["ODS"] >= Q1 - 1.5 * IQR) &
    (df["ODS"] <= Q3 + 1.5 * IQR)
].copy()

# Log transform
df_filt["log_ODS"] = np.log(df_filt["ODS"])

# Keep deterministic pair ordering
df_filt = df_filt.sort_values(
    by=["prompt_index", "img_name"]
)

# --------------------------------------------------------------
# Build prompt pairs
# --------------------------------------------------------------
paired = (
    df_filt
    .groupby("prompt_index")
    .agg({
        "log_ODS": list,
        "ODS": list,
        "prompt": list
    })
    .reset_index()
)

# Keep only complete pairs
paired = paired[
    paired["log_ODS"].apply(len) == 2
].copy()

# --------------------------------------------------------------
# PART 1 — Continuous Second-Noun Advantage (SNA)
#
# Same metric as FNA:
# SNA = log(ODS_A) + log(ODS_B)
#
# Interpretation:
# positive -> first noun bias
# negative -> second noun bias
# --------------------------------------------------------------

paired["SNA"] = paired["log_ODS"].apply(
    lambda x: x[0] + x[1]
)

sna_values = paired["SNA"].values

# --------------------------------------------------------------
# One-sided Wilcoxon
#
# H0: median(SNA) >= 0
# H1: median(SNA) < 0
#
# Evidence for SECOND-NOUN bias
# --------------------------------------------------------------

stat_w, p_w = wilcoxon(
    sna_values,
    alternative="less"
)

# --------------------------------------------------------------
# 95% Bootstrap Confidence Interval (BCa)
# --------------------------------------------------------------

boot = bootstrap(
    (sna_values,),
    np.mean,
    confidence_level=0.95,
    method="BCa",
    random_state=42
)

ci_low = boot.confidence_interval.low
ci_high = boot.confidence_interval.high

mean_sna = np.mean(sna_values)

# --------------------------------------------------------------
# Rank-biserial correlation (effect size)
# --------------------------------------------------------------

n = len(sna_values)

rank_biserial = 1 - (
    (2 * stat_w) / (n * (n + 1))
)

# --------------------------------------------------------------
# PART 2 — Strong second-noun bias
#
# noun2 dominates in BOTH prompt variants
#
# ODS < 1 -> object_2 dominates
# --------------------------------------------------------------

paired["strong_second_noun_bias"] = paired["ODS"].apply(
    lambda x: (x[0] < 1) and (x[1] < 1)
)

second_bias_rate = paired[
    "strong_second_noun_bias"
].mean()

# --------------------------------------------------------------
# Reporting
# --------------------------------------------------------------

print("=" * 70)
print("              SECOND-NOUN BIAS ANALYSIS")
print("=" * 70)

print(f"Number of paired prompts analysed: {len(paired)}")

print("\n[1] CONTINUOUS SECOND-NOUN ADVANTAGE (SNA)")
print("-" * 70)

print(f"Mean SNA: {mean_sna:.4f}")
print(f"95% BCa Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")

print(f"Wilcoxon Signed-Rank Statistic: {stat_w:.4f}")
print(f"One-sided Wilcoxon p-value: {p_w:.5e}")

print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

print("\n[2] STRONG SECOND-NOUN BIAS RATE")
print("-" * 70)

print(
    f"Pairs where noun2 dominates in BOTH variants: "
    f"{second_bias_rate:.2%}"
)

# --------------------------------------------------------------
# Interpretation
# --------------------------------------------------------------

print("\nINTERPRETATION")
print("-" * 70)

if p_w < 0.05:
    print(
        "-> Statistically significant evidence "
        "for a second-noun bias."
    )
else:
    print(
        "-> No statistically reliable second-noun "
        "advantage detected."
    )

if ci_high < 0:
    print("-> Confidence interval is entirely negative.")
    print(
        "   Evidence is consistent with a stable "
        "second-noun privileging effect."
    )

elif ci_low > 0:
    print("-> Confidence interval is entirely positive.")
    print(
        "   Evidence instead suggests a first-noun tendency."
    )

else:
    print("-> Confidence interval overlaps zero.")
    print(
        "   No stable directional second-noun effect detected."
    )

# Effect size interpretation
abs_r = abs(rank_biserial)

if abs_r < 0.1:
    magnitude = "negligible"
elif abs_r < 0.3:
    magnitude = "small"
elif abs_r < 0.5:
    magnitude = "moderate"
else:
    magnitude = "large"

print(
    f"-> Effect size magnitude: {magnitude} "
    f"(rank-biserial r = {rank_biserial:.3f})"
)

print("=" * 70)